In [1]:
import numpy as np
import time
m = 0.15

In [2]:
def create_COO_from_A(matrix):
    rows = []
    cols = []
    val = []
    n_page = matrix.shape[0]
    
    for i in range(0, n_page):
        for j in range(0, n_page):
            if matrix[i, j] != 0:
                val.append(matrix[i, j])
                rows.append(i)
                cols.append(j)
               
    val_array = np.array(val, dtype=np.float32) 
    rows_array = np.array(rows, dtype=np.int32)
    cols_array = np.array(cols, dtype=np.int32)
    return val_array, rows_array, cols_array, n_page

In [3]:
def create_A_from_COO(val, row, col, n_page):
    mat = np.zeros((n_page, n_page), dtype=np.float32)
    for i in range(0, len(val)):
        mat[row[i], col[i]] += val[i]
    
    return mat

In [4]:
def create_diz_from_COO(out_degree, in_degree):
    dictionary = {}
    for i in range(0, len(out_degree)):
        dictionary[i] = [in_degree[i], out_degree[i]]
    
    return dictionary

In [5]:
def create_COO_from_dat(filename):
    pages_names = []
    val_list = []
    rows_list = []
    column_list = []
    
    print(f"Reading file {filename}")
    with open(filename, 'r', encoding='utf-8', errors='ignore') as filein:
        header = filein.readline().strip().split()
        n_page = int(header[0])
        print(f"num Pages: {n_page}")

        for i in range(0, n_page):
            line = filein.readline()
            line = line.strip().split()
            pages_names.append(line[1])
            
        for line in filein:
            try:
                line = line.strip().split()
                if len(line) == 2 and int(line[1]) != int(line[0]):
                    line = [int(line[0]) - 1, int(line[1]) - 1]
                    column_list.append(line[0])
                    rows_list.append(line[1])
            except ValueError:
                continue
                
    n_links = len(column_list)
    print(f"Actuals links: {n_links}")
    
    rows = np.array(rows_list, dtype=np.int32)
    columns = np.array(column_list, dtype=np.int32)
    
    out_degree = np.zeros(n_page, dtype=np.int32)
    np.add.at(out_degree, columns, 1)
    
    in_degree = np.zeros(n_page, dtype=np.int32)
    np.add.at(in_degree, rows, 1)
    
    for i in range(0, n_links):
        val_list.append(1/out_degree[columns[i]])
    val = np.array(val_list, dtype=np.float32)
    
    return val, rows, columns, out_degree, pages_names

In [6]:
# Function to create M starting from A with dangling nodes
def crete_m_with_dangling_nodes(A_input):
    shape_a = A_input.shape[0]

    mask_not_dangling_nodes_function = np.abs(np.sum(A_input, axis=0)-1) <= 0.0001
    S_function = np.full((shape_a, shape_a), 1/shape_a)

    S_function[:, mask_not_dangling_nodes_function] = S_function[:, mask_not_dangling_nodes_function] * m

    M_output = (1 - m)*A_input + S_function
    return M_output

<h1>Construction of link matrix and matrix M for graph 2.1 and graph 2.2</h1>
<h4>We will refer with index 1 to the graph 2.1 and with index 2 to the graph 2.2</h4>

In [7]:
n_pages_1 = 4

A_1 = np.array([[0, 0, 1, 1/2],
                [1/3, 0, 0, 0],
                [1/3, 1/2, 0, 1/2],
                [1/3, 1/2, 0, 0]])
values1, row_index1, column_index1, _ = create_COO_from_A(A_1)
out_degree1 = np.zeros(n_pages_1, dtype=np.int32)
np.add.at(out_degree1, column_index1, 1)
print(out_degree1)

S_1 = np.full((n_pages_1, n_pages_1), 1/n_pages_1)
M_1 = (1 - m)*A_1 + m*S_1

print(M_1)

[3 2 1 2]
[[0.0375     0.0375     0.8875     0.4625    ]
 [0.32083333 0.0375     0.0375     0.0375    ]
 [0.32083333 0.4625     0.0375     0.4625    ]
 [0.32083333 0.4625     0.0375     0.0375    ]]


In [8]:
n_pages_2 = 5

A_2 = np.array([[0, 1, 0, 0, 0],
                [1, 0, 0, 0, 0],
                [0, 0, 0, 1, 1/2],
                [0, 0, 1, 0, 1/2],
                [0, 0, 0, 0, 0]])
values2, row_index2, column_index2, _  = create_COO_from_A(A_2)
out_degree2 = np.zeros(n_pages_2, dtype=np.int32)
np.add.at(out_degree2, column_index2, 1)
print(out_degree2)

S_2 = np.full((n_pages_2, n_pages_2), 1/n_pages_2)
M_2 = (1 - m)*A_2 + m*S_2

print(M_2)

[1 1 1 1 2]
[[0.03  0.88  0.03  0.03  0.03 ]
 [0.88  0.03  0.03  0.03  0.03 ]
 [0.03  0.03  0.03  0.88  0.455]
 [0.03  0.03  0.88  0.03  0.455]
 [0.03  0.03  0.03  0.03  0.03 ]]


<h1>Construction of Link Matrix A in the COO format</h1>

In [9]:
file_input = "hollins.dat" 
val_hollins, row_hollins, col_hollins, out_degree_hollins, page_names = create_COO_from_dat(file_input)
num_pages_hollins = out_degree_hollins.shape[0]

in_degree_hollins = np.zeros(num_pages_hollins, dtype=np.int32)
np.add.at(in_degree_hollins, row_hollins, 1)

A_hollins = create_A_from_COO(val_hollins, row_hollins, col_hollins, out_degree_hollins.shape[0])
diz = create_diz_from_COO(out_degree_hollins, in_degree_hollins)

Reading file hollins.dat
num Pages: 6012
Actuals links: 23875


<h1>Power Method</h1>

In [10]:
def power_method(row_index, column_index, out_degree, max_iter=100, tol=1.0e-8, m = 0.15):
    # Finds the dangling nodes
    dangling_mask = (out_degree == 0)
    
    # x guess vector initialization
    n_pages = out_degree.shape[0]
    x = np.ones(n_pages, dtype=np.float64) / n_pages
    
    # m*S
    randomness = m / n_pages
    e_current = 0
    
    print("Starting page rank:")
    start_time = time.time()
    for i in range(0, max_iter):
        x_prev = x.copy()
        
        # It's like doing A*x since that Aij = 1/out_degree(ij)
        first_comp = x[column_index] / out_degree[column_index]
        
        # Effective implementation of x = (1 - m)*Ax
        x = np.zeros(n_pages, dtype=np.float64)
        np.add.at(x, row_index, first_comp)
        x *= 1 - m
        
        # Final computation x = x + mS with attention to the dangling nodes
        dangling_sum = np.sum(x_prev[dangling_mask])
        rank_correction = randomness + ((1 - m) * dangling_sum / n_pages)
        x += rank_correction
        
        e_current = (x_prev@x) / (x_prev@x_prev)
        
        err = np.sum(np.abs(x - x_prev))
        if err < tol:
            print(f"Converged at iteration {i}.")
            break
                
    print(f"Finished in {time.time() - start_time:.5f} seconds")
    return e_current, x

<h1>Results Graph 2.1</h1>

In [12]:
start_time = time.time()
e1, v1 = np.linalg.eig(M_1)
v_calc1 = v1[:, np.argmax(e1.real)].real
v_calc1 = v_calc1/np.sum(v_calc1)
print(f"Finished in {time.time() - start_time:.5f} seconds")

Finished in 0.00100 seconds


In [13]:
e_guess1, v_guess1 = power_method(row_index1, column_index1, out_degree1)

Starting page rank:
Convergenza raggiunta all'iterazione 23.
Finished in 0.00151 seconds


In [14]:
print(f"Eigenvectors of M_1:\n{'Found with np.linalg.eig: ':<29}{v_calc1}")
print(f"Found with the power method: {v_guess1}")
print(f"Eigenvalue with power method: {e_guess1}")
print(f"Error: {np.linalg.norm(v_guess1 - v_calc1)}")

Eigenvectors of M_1:
Found with np.linalg.eig:    [0.36815068 0.14180936 0.28796163 0.20207834]
Found with the power method: [0.36815068 0.14180936 0.28796163 0.20207834]
Eigenvalue with power method: 0.9999999997656641
Error: 1.7445149539871232e-09


<h1>Results Graph 2.2</h1>

In [15]:
# Graph 2.2
start_time = time.time()
e2, v2 = np.linalg.eig(M_2)
v_calc2 = v2[:, np.argmax(e2.real)].real
v_calc2 = v_calc2/np.sum(v_calc2)
print(f"Finished in {time.time() - start_time:.5f} seconds")

Finished in 0.00100 seconds


In [16]:
e_guess2, v_guess2 = power_method(row_index2, column_index2, out_degree2)

Starting page rank:
Convergenza raggiunta all'iterazione 1.
Finished in 0.00000 seconds


In [17]:
print(f"Eigenvectors of M_2:\n{'Found with np.linalg.eig: ':<29}{v_calc2}")
print(f"Found with the power method: {v_guess2}")
print(f"Eigenvalue with power method: {e_guess2}")
print(f"Error: {np.linalg.norm(v_guess2 - v_calc2)}")

Eigenvectors of M_2:
Found with np.linalg.eig:    [0.2   0.2   0.285 0.285 0.03 ]
Found with the power method: [0.2   0.2   0.285 0.285 0.03 ]
Eigenvalue with power method: 1.0
Error: 1.7627658722935884e-15


<h1>Results Hollins Database</h1>

In [18]:
M_hollins = crete_m_with_dangling_nodes(A_hollins)

In [19]:
start_time = time.time()
e_hollins, v_hollins = np.linalg.eig(M_hollins)
ind_max = np.argmax(e_hollins)
v_calc_hollins = v_hollins[:, ind_max].real/np.sum(v_hollins[:, ind_max].real)
print(f"Finished in {time.time() - start_time:.5f} seconds")

Finished in 83.06394 seconds


In [20]:
e_guess_hollins, v_guess_hollins = power_method(row_hollins, col_hollins, out_degree_hollins)

Starting page rank:
Convergenza raggiunta all'iterazione 83.
Finished in 0.07628 seconds


In [21]:
print(f"Eigenvectors of M_hollins:\n{'Found with np.linalg.eig: ':<29}\n{v_calc_hollins}")
print(f"Found with the power method:\n{v_guess_hollins}")
print(f"Eigenvalue with power method: {e_guess_hollins}")
print(f"Error: {np.linalg.norm(v_guess_hollins - v_calc_hollins)}")

Eigenvectors of M_hollins:
Found with np.linalg.eig:    
[5.80584118e-05 1.98787505e-02 1.12567975e-04 ... 6.89072123e-05
 6.89072123e-05 1.75820619e-04]
Found with the power method:
[5.80584153e-05 1.98787511e-02 1.12567981e-04 ... 6.89072159e-05
 6.89072159e-05 1.75820618e-04]
Eigenvalue with power method: 0.999999996165265
Error: 2.7296176611360708e-09


In [22]:
sorted_indices = np.argsort(v_guess_hollins)

In [23]:
n_top = 10
top_n_indices = sorted_indices[-n_top:][::-1]

print(f"Page ranking, Top {n_top}:")
for i in range(len(top_n_indices)):
    print(f"{" ":<15} {"Score:":<11} {v_guess_hollins[top_n_indices[i]]}")
    print(f"{i+1:>4}. Page {top_n_indices[i]+1:<4} Pointed by: {diz[top_n_indices[i]][0]}")
    print(f"{" ":<15} {"Points to:":<11} {diz[top_n_indices[i]][1]}\n")

Page ranking, Top 10:
                Score:      0.019878751073977855
   1. Page 2    Pointed by: 829
                Points to:  25

                Score:      0.00928762058003152
   2. Page 37   Pointed by: 454
                Points to:  14

                Score:      0.00861039323889531
   3. Page 38   Pointed by: 435
                Points to:  31

                Score:      0.008065030976292962
   4. Page 61   Pointed by: 390
                Points to:  10

                Score:      0.008026565147472783
   5. Page 52   Pointed by: 417
                Points to:  11

                Score:      0.007164643207519007
   6. Page 43   Pointed by: 377
                Points to:  15

                Score:      0.006582780905201322
   7. Page 425  Pointed by: 87
                Points to:  77

                Score:      0.0059892133080154016
   8. Page 27   Pointed by: 168
                Points to:  12

                Score:      0.0055717362685195025
   9. Page 28   Pointed by

In [24]:
n_bottom = 10
bottom_n_indices = sorted_indices[:n_bottom]

print(f"Page ranking, Bottom {n_bottom}:")
for i in reversed(range(len(bottom_n_indices))):
    print(f"{" ":<15} {"Score:":<11} {v_guess_hollins[bottom_n_indices[i]]}")
    print(f"{num_pages_hollins - i:>4}. Page {bottom_n_indices[i]+1:<4} Pointed by: {diz[bottom_n_indices[i]][0]}")
    print(f"{" ":<15} {"Points to:":<11} {diz[bottom_n_indices[i]][1]}\n")

Page ranking, Bottom 10:
                Score:      5.843605142657109e-05
6003. Page 2021 Pointed by: 1
                Points to:  0

                Score:      5.843605142657109e-05
6004. Page 2020 Pointed by: 1
                Points to:  0

                Score:      5.843605142657109e-05
6005. Page 2019 Pointed by: 1
                Points to:  0

                Score:      5.843605142657109e-05
6006. Page 2018 Pointed by: 1
                Points to:  0

                Score:      5.843605142657109e-05
6007. Page 2017 Pointed by: 1
                Points to:  0

                Score:      5.843605142657109e-05
6008. Page 2016 Pointed by: 1
                Points to:  0

                Score:      5.843605142657109e-05
6009. Page 2015 Pointed by: 1
                Points to:  0

                Score:      5.843605142657109e-05
6010. Page 2014 Pointed by: 1
                Points to:  0

                Score:      5.805841534045088e-05
6011. Page 51   Pointed by: 0
       

<h1>Power method with huge web</h1>
<h4>This web is created with 900000 unique pages and 3600000 links randomly created between the pages, avoiding a page pointing to itself</h4>

In [25]:
NUM_PAGES = 900000
NUM_LINKS = 3600000
OUTPUT_FILE = "large_web_graph.dat"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(f"{NUM_PAGES} {NUM_LINKS}\n")
    
    base_url = "http://www.fakesite.com/page_"

    for i in range(1, NUM_PAGES + 1):
        url = f"{base_url}{i}.html"
        f.write(f"{i} {url}\n")

    for _ in range(NUM_LINKS):
        source = np.random.randint(1, NUM_PAGES)
        target = np.random.randint(1, NUM_PAGES)

        while source == target:
            target = np.random.randint(1, NUM_PAGES)

        f.write(f"{source} {target}\n")

In [26]:
file_input = "large_web_graph.dat" 
val_large, row_large, col_large, out_degree_large, _ = create_COO_from_dat(file_input)
num_pages_large = out_degree_large.shape[0]

in_degree_large = np.zeros(num_pages_large, dtype=np.int32)
np.add.at(in_degree_large, row_large, 1)

Reading file large_web_graph.dat
num Pages: 900000
Actuals links: 3600000


In [27]:
e_guess_large, v_guess_large = power_method(row_large, col_large, out_degree_large)
print(e_guess_large)

Starting page rank:
Convergenza raggiunta all'iterazione 24.
Finished in 3.29026 seconds
0.9999999999936742
